In [1]:
import os
from getpass import getpass

os.environ["GH_TOKEN"] = getpass("Paste GitHub token: ")
os.environ["HF_TOKEN"] = getpass("Paste Hugging Face token: ")

In [2]:
%cd /content
!rm -rf vnlegal-rag-v2
!git clone https://${GH_TOKEN}@github.com/hyperformancelabs/vnlegal-rag-v2.git
%cd vnlegal-rag-v2
!pip install -e .

/content
Cloning into 'vnlegal-rag-v2'...
remote: Enumerating objects: 430, done.
remote: Counting objects: 100% (322/322), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 430 (delta 185), reused 289 (delta 161), pack-reused 108 (from 2)
Receiving objects: 100% (430/430), 121.41 MiB | 16.26 MiB/s, done.
Resolving deltas: 100% (216/216), done.
/content/vnlegal-rag-v2
Obtaining file:///content/vnlegal-rag-v2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 128.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 146.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 140.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━

In [3]:
!git pull

Already up to date.


In [ ]:
!gdown 1vkAnfAHNPEhu-aKrhR17RbyY3TYqXLzm
!mkdir -p data/processed
!unzip -o data_processed.zip -d data/processed
!rm -rf data/processed/__MACOSX data_processed.zip
!ls data/processed/

Downloading...
From (original): https://drive.google.com/uc?id=1vkAnfAHNPEhu-aKrhR17RbyY3TYqXLzm
From (redirected): https://drive.google.com/uc?id=1vkAnfAHNPEhu-aKrhR17RbyY3TYqXLzm&confirm=t&uuid=14f6caf5-05e9-4d15-9653-5f9ef0da154d
To: /content/vnlegal-rag-v2/data_processed.zip
100% 360M/360M [00:01<00:00, 263MB/s] 
Archive:  data_processed.zip
  inflating: data/processed/corpus.csv  
  inflating: data/processed/corpus_pyvi.csv  
  inflating: data/processed/__MACOSX/._corpus_pyvi.csv  
  inflating: data/processed/train.csv  
  inflating: data/processed/eval.csv  
  inflating: data/processed/train_pyvi.csv  
  inflating: data/processed/eval_pyvi.csv  
  inflating: data/processed/train_underthesea.csv  
  inflating: data/processed/eval_underthesea.csv  
  inflating: data/processed/corpus_underthesea.csv  
corpus.csv		eval.csv	      train.csv
corpus_pyvi.csv		eval_pyvi.csv	      train_pyvi.csv
corpus_underthesea.csv	eval_underthesea.csv  train_underthesea.csv


In [5]:
!pip install -U huggingface_hub
!hf auth login --token ${HF_TOKEN}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.4/684.4 kB 47.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.17.0
    Uninstalling huggingface_hub-1.17.0:
      Successfully uninstalled huggingface_hub-1.17.0
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `base` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [6]:
# Set batch sizes for target GPU
# T4=16GB, L4=24GB, A100=40GB
!python scripts/update_batch_size.py --vram auto

Target: 23GB VRAM

  Model                                            |  enc | rerk | trBi |  trX | cap
  ------------------------------------------------ | ---- | ---- | ---- | ---- | ---
  multilingual-e5-base                             |  128 |   64 |   32 |   16 | 128
  vietnamese-bi-encoder                            |  256 |  128 |  128 |   64 | 256
  granite-embedding-97m-multilingual-r2            |  128 |   64 |   64 |   32 | 128
  embeddinggemma-300m                              |   64 |   32 |   64 |   32 | 64
  PhoRanker                                        |  256 |  128 |  128 |   64 | 256


configs/dense-selection/:
  e5-base.yaml: encode → bs=128 (already set)
  embeddinggemma-300m.yaml: encode → bs=64 (already set)
  granite-97m.yaml: encode → bs=128 (already set)
  vietnamese-bi-encoder.yaml: encode → bs=256 (already set)

configs/model-selection/:
  embeddinggemma-300m.yaml: encode → bs=64 (already set)
  ⚠ f2llm-v2-160m.yaml: unknown model 'codefuse-ai/F2LLM-v2-16

In [7]:
!python scripts/run_pipeline.py \
    configs/dense-selection/*.yaml --results \
    results/scores.json

modules.json: 100% 387/387 [00:00<00:00, 1.77MB/s]
README.md: 100% 179k/179k [00:00<00:00, 68.8MB/s]
sentence_bert_config.json: 100% 57.0/57.0 [00:00<00:00, 251kB/s]
config.json: 100% 694/694 [00:00<00:00, 3.16MB/s]
model.safetensors: 100% 1.11G/1.11G [00:02<00:00, 381MB/s] 
Loading weights: 100% 199/199 [00:00<00:00, 4422.61it/s]
tokenizer_config.json: 100% 418/418 [00:00<00:00, 2.10MB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:01<00:00, 4.78MB/s]
tokenizer.json: 100% 17.1M/17.1M [00:00<00:00, 51.6MB/s]
special_tokens_map.json: 100% 280/280 [00:00<00:00, 1.32MB/s]
config.json: 100% 200/200 [00:00<00:00, 1.41MB/s]
Batches: 100% 2049/2049 [35:42<00:00,  1.05s/it]
Batches: 100% 105/105 [00:09<00:00, 11.02it/s]

Running: dense-multilingual-e5-base
  Queries: 13364  |  Corpus: 262168
  mrr@10: 0.4115
  success@100: 0.8760
  Scores → results/dense-selection/e5-base-scores.json
modules.json: 100% 573/573 [00:00<00:00, 2.46MB/s]
config_sentence_transformers.json: 100% 997/997 [00:00<00:

In [9]:
!cat results/scores.json | python3 -m json.tool

{
    "bm25-non-seg_k1=1.0_b=0.6": {
        "mrr@10": 0.32703294327808013,
        "success@100": 0.7790332235857528
    },
    "bm25-non-seg_k1=1.0_b=0.75": {
        "mrr@10": 0.35213535653710765,
        "success@100": 0.7906315474408859
    },
    "bm25-non-seg_k1=1.0_b=0.9": {
        "mrr@10": 0.36137119743637286,
        "success@100": 0.7974408859622868
    },
    "bm25-non-seg_k1=1.2_b=0.6": {
        "mrr@10": 0.32938930935040023,
        "success@100": 0.7832235857527686
    },
    "bm25-non-seg_k1=1.2_b=0.75": {
        "mrr@10": 0.3548417211841345,
        "success@100": 0.7955701885662975
    },
    "bm25-non-seg_k1=1.2_b=0.9": {
        "mrr@10": 0.36295778637704673,
        "success@100": 0.8044747081712063
    },
    "bm25-non-seg_k1=1.5_b=0.6": {
        "mrr@10": 0.32886655335585363,
        "success@100": 0.7878629152948219
    },
    "bm25-non-seg_k1=1.5_b=0.75": {
        "mrr@10": 0.355086307445257,
        "success@100": 0.8002095181083508
    },
    "bm25-non-

In [18]:
!git status

On branch master
Your branch is up to date with 'origin/master'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   results/dense-selection/e5-base-scores.json
	new file:   results/dense-selection/embeddinggemma-300m-scores.json
	new file:   results/dense-selection/granite-97m-scores.json
	new file:   results/dense-selection/vietnamese-bi-encoder-scores.json
	modified:   results/scores.json



In [24]:
!git config user.email "vhphatdz@gmail.com"
!git config user.name "phatv1"
!git commit -m "results: dense selection zero-shot evaluation"
!git push

[master 86d634f] results: dense selection zero-shot evaluation
 5 files changed, 40 insertions(+)
 create mode 100644 results/dense-selection/e5-base-scores.json
 create mode 100644 results/dense-selection/embeddinggemma-300m-scores.json
 create mode 100644 results/dense-selection/granite-97m-scores.json
 create mode 100644 results/dense-selection/vietnamese-bi-encoder-scores.json
Enumerating objects: 12, done.
Counting objects: 100% (12/12), done.
Delta compression using up to 12 threads
Compressing objects: 100% (9/9), done.
Writing objects: 100% (9/9), 1.25 KiB | 1.25 MiB/s, done.
Total 9 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/hyperformancelabs/vnlegal-rag-v2.git
   00121e4..86d634f  master -> master
